## CASH Notebook

## Celestial Chase - LVL 3: The Star Chart

You are alone. 40 light-years from Earth. The Hail Mary is your last hope.

Mission control's final message was not sent in words. It was written in the stars themselves.

They ranked every visible star by how high it stood in the sky. The brightest in altitude came first. They marked 5 of them red - one per letter. The redness tells you the shift. The rank tells you the order.

Find the red stars. Measure their glow. Undo the shifts. The word will appear.

---

**The encoded signal:** `ysult`

**Your task:**
1. Display the star chart. The **red pixels** carry the message - filter by `B == 0` and `G == 0` and `R >= 28`. Decoy red pixels have `R < 28`.
2. Use `ephem` to compute the **altitude** of all 15 stars on `2025/6/21 12:00:00` UTC from Zurich:
   ```python
   stars = ['Sirius', 'Canopus', 'Arcturus', 'Vega', 'Capella', 'Rigel', 'Procyon', 'Betelgeuse', 'Altair', 'Aldebaran', 'Antares', 'Spica', 'Pollux', 'Fomalhaut', 'Deneb']
   ```
3. Sort all 15 stars by altitude **descending**. Take the top **5** - these are the message stars, in order.
4. For each of the top 5 stars (in altitude-rank order), find its pixel in the chart and read the **red channel**: `img[y, x, 2]`
5. **Reverse the Caesar shift** for each letter `i`: `decoded[i] = (encoded[i] - red_channel[i] % 26) % 26`
6. The transposition is the identity permutation - so after reversing shifts the word is already in order.

**Position formula:**
```python
x = int((az_deg % 360) / 360 * 800) % 800
y = int((90 - alt_deg) / 180 * 800) % 800
```


In [ ]:
import base64, cv2, numpy as np
from IPython.display import display, Image as IPImage

img_b64 = "..."

img_bytes = base64.b64decode(img_b64)
img_arr   = np.frombuffer(img_bytes, dtype=np.uint8)
img       = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)

cv2.imwrite('starchart.png', img)
display(IPImage('starchart.png'))
print("Image shape:", img.shape)


In [ ]:
import ephem, math, numpy as np

encoded    = "ysult"
obs_date   = "2025/6/21 12:00:00"
obs_lat    = "47.3769"
obs_lon    = "8.5417"
n          = 5
W, H       = 800, 800
star_names = ['Sirius', 'Canopus', 'Arcturus', 'Vega', 'Capella', 'Rigel', 'Procyon', 'Betelgeuse', 'Altair', 'Aldebaran', 'Antares', 'Spica', 'Pollux', 'Fomalhaut', 'Deneb']

# TODO Step 1: Compute az and alt for each star
obs = ephem.Observer()
obs.lat  = obs_lat
obs.lon  = obs_lon
obs.date = obs_date

star_data = []
for name in star_names:
    star = ephem.star(name)
    star.compute(obs)
    az_deg  = int(math.degrees(float(star.az)))
    alt_deg = int(math.degrees(float(star.alt)))
    star_data.append((name, az_deg, alt_deg))

# TODO Step 2: Sort by altitude descending, take top n
sorted_stars = sorted(star_data, key=lambda s: s[2], reverse=True)
top_stars    = sorted_stars[:n]

# TODO Step 3: For each top star compute pixel position and read red channel from img
# x = int((az_deg % 360) / 360 * W) % W
# y = int((90 - alt_deg) / 180 * H) % H
# red = img[y, x, 2]
red_channels = []  # fill this in - should have n values

# TODO Step 4: Reverse the Caesar shifts
answer = ""
for i, c in enumerate(encoded):
    shift = red_channels[i] % 26
    answer += chr((ord(c) - ord('a') - shift) % 26 + ord('a'))

print(answer)
